In [ ]:
# ===== ONE-CELL: Exact hash + perceptual hash + clean folder (NO leakage) =====

# 0) Install deps (safe to run repeatedly)
import sys, subprocess
def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg])
for _p in ["pillow", "imagehash", "tqdm", "pandas"]:
    try:
        __import__(_p if _p != "pillow" else "PIL")
    except Exception:
        _pip(_p)

# 1) Imports
from pathlib import Path
import shutil, hashlib
import pandas as pd
from PIL import Image, ImageFile
import imagehash
from tqdm import tqdm
from collections import defaultdict
from itertools import combinations

ImageFile.LOAD_TRUNCATED_IMAGES = True

# 2) CONFIG  ✅ change ROOT if needed
ROOT = Path(r"C:\Users\cse.majib\Desktop\Data sets\New folder\New folder")  # folder containing class subfolders
OUT_ROOT = ROOT.parent / (ROOT.name + "_CLEAN_NO_LEAK")

IMG_EXT = {".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp"}

USE_EXACT_SHA256 = True
USE_PIXEL_HASH   = True
USE_PERCEPTUAL_PHASH = True

PIXEL_HASH_SIZE = (256, 256)

PHASH_HASH_SIZE = 8      # 8 => 64-bit pHash
PHASH_MAX_DIST  = 3      # 0 exact only; 1-3 strict; higher = more aggressive

print("ROOT:", ROOT)
print("OUT_ROOT:", OUT_ROOT)

# 3) Helpers
def label_of(root: Path, p: Path) -> str:
    rel = p.relative_to(root)
    return rel.parts[0] if len(rel.parts) else ""

def list_images(root: Path):
    return [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT]

def sha256_file(p: Path, chunk_size=1024*1024) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b: break
            h.update(b)
    return h.hexdigest()

def pixel_md5(p: Path) -> str:
    with Image.open(p) as im:
        im = im.convert("RGB").resize(PIXEL_HASH_SIZE)
        return hashlib.md5(im.tobytes()).hexdigest()

def phash_hex(p: Path) -> str:
    with Image.open(p) as im:
        im = im.convert("RGB")
        return str(imagehash.phash(im, hash_size=PHASH_HASH_SIZE))

def hamming_dist_hex(a: str, b: str) -> int:
    return (int(a,16) ^ int(b,16)).bit_count()

# Union-Find
class DSU:
    def __init__(self):
        self.parent = {}
        self.rank = {}
    def add(self, x):
        if x not in self.parent:
            self.parent[x] = x
            self.rank[x] = 0
    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1

# LSH buckets for 64-bit pHash (hash_size=8)
def lsh_buckets(ph: str):
    s = ph.lower().rjust(16, "0")  # 64-bit => 16 hex chars
    return [("b0", s[0:4]), ("b1", s[4:8]), ("b2", s[8:12]), ("b3", s[12:16])]

# 4) Scan + hash
imgs = list_images(ROOT)
print("Total images found:", len(imgs))

dsu = DSU()
sha_groups = defaultdict(list)
pix_groups = defaultdict(list)
ph_map = {}
rows = []

for p in tqdm(imgs, desc="Hashing"):
    p = p.resolve()
    sp = str(p)
    dsu.add(sp)
    lab = label_of(ROOT, p)

    s256 = ""
    pmd5 = ""
    ph   = ""

    if USE_EXACT_SHA256:
        try:
            s256 = sha256_file(p)
            sha_groups[s256].append(sp)
        except Exception:
            pass

    if USE_PIXEL_HASH:
        try:
            pmd5 = pixel_md5(p)
            pix_groups[pmd5].append(sp)
        except Exception:
            pass

    if USE_PERCEPTUAL_PHASH:
        try:
            ph = phash_hex(p)
            ph_map[sp] = ph
        except Exception:
            pass

    try:
        bsz = p.stat().st_size
    except Exception:
        bsz = 0

    rows.append({
        "path": sp,
        "label": lab,
        "sha256": s256,
        "pixel_md5": pmd5,
        "phash": ph,
        "bytes": bsz
    })

df = pd.DataFrame(rows)

# 5) Union edges: exact + pixel
exact_edges = 0
if USE_EXACT_SHA256:
    for h, paths in sha_groups.items():
        if len(paths) > 1:
            base = paths[0]
            for x in paths[1:]:
                dsu.union(base, x)
                exact_edges += 1

pixel_edges = 0
if USE_PIXEL_HASH:
    for h, paths in pix_groups.items():
        if len(paths) > 1:
            base = paths[0]
            for x in paths[1:]:
                dsu.union(base, x)
                pixel_edges += 1

print("Exact edges:", exact_edges, "| Pixel edges:", pixel_edges)

# 6) Union edges: near duplicates by pHash (LSH)
near_edges = 0
near_pairs_checked = 0

if USE_PERCEPTUAL_PHASH and PHASH_HASH_SIZE == 8:
    bucket = defaultdict(list)
    for path, ph in ph_map.items():
        for band, key in lsh_buckets(ph):
            bucket[(band, key)].append(path)

    seen = set()
    for _, paths in tqdm(bucket.items(), desc="pHash buckets"):
        if len(paths) < 2:
            continue
        for a, b in combinations(paths, 2):
            key = tuple(sorted((a,b)))
            if key in seen:
                continue
            seen.add(key)
            near_pairs_checked += 1
            da = ph_map.get(a, "")
            db = ph_map.get(b, "")
            if not da or not db:
                continue
            if hamming_dist_hex(da, db) <= PHASH_MAX_DIST:
                dsu.union(a, b)
                near_edges += 1

print("Near edges (pHash):", near_edges, "| pHash pairs checked:", near_pairs_checked)

# 7) Build components
label_map = dict(zip(df["path"], df["label"]))
size_map  = dict(zip(df["path"], df["bytes"]))

comp = defaultdict(list)
for p in df["path"].tolist():
    comp[dsu.find(p)].append(p)

# 8) Decide keep/drop (NO leakage)
keep = set()
drop_reason = {}

for root, paths in comp.items():
    labels = {label_map.get(p,"") for p in paths}
    labels.discard("")
    if len(labels) > 1:
        # Cross-class dup cluster => drop all
        for p in paths:
            drop_reason[p] = "CROSS_CLASS_DUP_CLUSTER"
        continue

    # Within one class => keep best rep only
    paths_sorted = sorted(paths, key=lambda x: (size_map.get(x,0), x), reverse=True)
    rep = paths_sorted[0]
    keep.add(rep)
    for p in paths_sorted[1:]:
        drop_reason[p] = "DUP_WITHIN_CLASS_CLUSTER"

print("Keep:", len(keep), "| Drop:", len(drop_reason))

# 9) Create output + copy
OUT_ROOT.mkdir(parents=True, exist_ok=True)

copied = 0
for sp in tqdm(sorted(keep), desc="Copying kept files"):
    src = Path(sp)
    lab = label_map.get(sp, "unknown")
    dst_dir = OUT_ROOT / lab
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / src.name
    if dst.exists():
        dst = dst_dir / f"{src.stem}__dupname{src.suffix}"
    shutil.copy2(src, dst)
    copied += 1

# 10) Reports
report_dir = OUT_ROOT / "_reports"
report_dir.mkdir(exist_ok=True)

df["status"] = df["path"].apply(lambda x: "KEEP" if x in keep else "DROP")
df["drop_reason"] = df["path"].map(drop_reason).fillna("")
df.to_csv(report_dir / "hash_and_decisions.csv", index=False)

summary = (
    df.groupby(["label","status"])["path"]
      .count()
      .reset_index()
      .rename(columns={"path":"count"})
)
summary.to_csv(report_dir / "summary_by_class.csv", index=False)

print("\n✅ DONE")
print("✅ Clean dataset:", OUT_ROOT)
print("✅ Copied:", copied)
print("✅ Reports:", report_dir)
display(summary)
